In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import regexp_replace, col, when, nullif
import matplotlib.pyplot as plt

1. Data dictionary (some datasets have hundreds of records, so focus on the data that passes your missing value analysis).
2. Data size and source.
3. How are you digesting the data and your checkpointing strategy.
4. Missing and duplicate value plan and analysis.
5. Quick and dirty EDA (summary statistics, etc.)

## Helper functions

In [0]:
def get_null_rows(df):
  '''Display number of null records for each column in the dataframe.'''
  return df.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df.columns]).transpose()

def get_null_info(df):
    '''Display number of null records and null percentage for each column in the dataframe'''
    total_rows = df.count()

    exprs = [
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]
    counts = df.select(exprs)

    # Convert wide → long
    long_df = counts.select(
        F.explode(F.map_from_arrays(
            F.array(*[F.lit(c) for c in df.columns]),
            F.array(*[F.col(c) for c in df.columns])
        )).alias("column", "null_count")
    )

    # Add percentage
    return long_df.withColumn(
        "null_pct", 
        F.concat(
            F.round((F.col("null_count") / total_rows)*100, 2).cast("string"),
            F.lit("%")
        )
    )

def get_shape(df):
  '''Return the shape of the dataframe'''
  return f"{df.count()} rows, {len(df.columns)} columns"

def get_categorical_columns(df):
  '''Return a list of categorical columns'''
  return [c for (c, t) in df.dtypes if t in ('string', 'boolean')]


In [0]:
# Create checkpoint folder
section = "03"
number = "02"
folder_path = f"dbfs:/student-groups/Group_{section}_{number}"
# dbutils.fs.mkdirs(folder_path)



# EDA for Weather Dataset

In [0]:
# Full weather data
df_weather_full = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data/")

# Num rows/cols
print(get_shape(df_weather_full))

In [0]:
# Weather Data 3months
df_weather_3m = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data_3m/")
display(df_weather_3m)

In [0]:
df_weather_categorical_cols = get_categorical_columns(df_weather_3m)
df_weather_categorical_cols

Numerical columns show up as categorical (i.e 'Hourly*'), which tells us that we need to clean up some values in these columns.

## Display unique values for each hourly weather variable to get idea of what to clean

In [0]:
# display(df_weather_3m.select("HourlyPrecipitation").distinct()) # Trace amounts of precipitation are indicated with a “T.”. Measured in inches to the nearest hundredth.
# display(df_weather_3m.select("HourlyRelativeHumidity").distinct())
# display(df_weather_3m.select("HourlyVisibility").distinct())
# display(df_weather_3m.select("HourlyAltimeterSetting").distinct())
# display(df_weather_3m.select("HourlySkyConditions").distinct())
display(df_weather_3m.select("HourlyStationPressure").distinct())

In [0]:
# Num rows/cols
print(get_shape(df_weather_3m))

In [0]:
df_weather_3m.printSchema()

## Count number of null records for each column

In [0]:
null_info_df_weather_3m = get_null_info(df_weather_3m)
display(null_info_df_weather_3m)

In [0]:
#null_info_df_weather_3m.write.parquet(f"{folder_path}/null_info_df_weather_3m.parquet")
null_info_df_weather_3m = spark.read.parquet(f"{folder_path}/null_info_df_weather_3m.parquet")

In the weather dataset, there are different levels of granularity in which they are measuring the weather data (i.e. hourly, daily, monthly).


## Ensure there are no duplicate rows in weather data

In [0]:
assert(df_weather_3m.count() == df_weather_3m.dropDuplicates().count())

In [0]:
cols_to_keep = ['Station', 'Date', 'REPORT_TYPE', 'Source', 'Year', 'Latitude', 'Longitude']
hourly_cols_lt_50_pct_null = ['HourlyDryBulbTemperature', 'HourlyWindSpeed', 'HourlyWindDirection', 'HourlyDewPointTemperature', 'HourlyRelativeHumidity', 'HourlyVisibility', 'HourlyAltimeterSetting', 'HourlySkyConditions', 'HourlyStationPressure']

df_weather_hourly = df_weather_3m.select(cols_to_keep + hourly_cols_lt_50_pct_null).dropna()

display(df_weather_hourly)

In [0]:
display(df_weather_hourly.summary())

In [0]:
print(get_shape(df_weather_hourly))

In [0]:
# df_weather_hourly.write.parquet(f"{folder_path}/df_weather_hourly.parquet")
df_weather_hourly = spark.read.parquet(f"{folder_path}/df_weather_hourly.parquet")

After removing all null values, the hourly weather data has ~~15889~~ 8459585 records.
We still need to clean up bad values in our hourly data.

# Get Hourly weather variables that are casted as categorical columns (even if they are numerical)

In [0]:
# df_weather_categorical_cols = get_categorical_columns(df_weather_3m)
hourly_cat_vars = list(set([c.lower() for c in df_weather_hourly.columns if 'hourly' in c.lower()]).intersection(set([c.lower() for c in df_weather_hourly.columns])))
hourly_cat_vars

# Replace extraneous values using regex, then cast column to appropriate data type

In [0]:
# Define regex strings for each col to remove extraneous values in hourly categorical columns before converting to specified data types
hourly_cat_vars_to_regex_types = {
    'HourlyStationPressure': ['s', '', 'double'],
    'HourlyDewPointTemperature': ["[^0-9\.-]", '', 'int'],
    'HourlyDryBulbTemperature': ["[^0-9\.-]", '', 'int'],
    'HourlyWindSpeed': ['s', '', 'int'],
    'HourlyVisibility': ['[^0-9\.]', '', 'double'],
    'HourlyRelativeHumidity': ['\*', '0', 'int'],
    'HourlyAltimeterSetting': ['s', '', 'double'],
    'HourlyWindDirection': ["[^0-9]", '', 'int']
}

In [0]:
# dict_keys(['HourlyStationPressure', 'HourlyDewPointTemperature', 'HourlyDryBulbTemperature', 'HourlyWindSpeed', 'HourlyVisibility', 'HourlyRelativeHumidity', 'HourlyAltimeterSetting', 'HourlyWindDirection'])
# hourly_cat_vars_to_regex.keys()
display(df_weather_hourly.select(list(hourly_cat_vars_to_regex_types.keys())[6]).distinct())

In [0]:
df_weather_hourly = df_weather_hourly.replace('*', '0')
df_weather_hourly = df_weather_hourly.replace('2237', '0', subset = 'HourlyWindSpeed')

for col_name, payload in hourly_cat_vars_to_regex_types.items():
  regex_str, replace_with, cast_type = payload
  df_weather_hourly = df_weather_hourly.withColumn(col_name, regexp_replace(col_name, regex_str, replace_with).try_cast(cast_type))

display(df_weather_hourly.select(list(hourly_cat_vars_to_regex_types.keys())).summary())

In [0]:
numeric_cols = [
    "HourlyDryBulbTemperature", "HourlyDewPointTemperature", 
    "HourlyWindSpeed", "HourlyVisibility", "HourlyStationPressure"
]

ax = df_weather_hourly.plot.box(column=numeric_cols[0])
ax.update_layout(title=f"Summary Statistics of {numeric_cols[0]}")

In [0]:
ax = df_weather_hourly.plot.box(column=numeric_cols[1])
ax.update_layout(title=f"Summary Statistics of {numeric_cols[1]}")

In [0]:
ax = df_weather_hourly.plot.box(column=numeric_cols[2])
ax.update_layout(title=f"Summary Statistics of {numeric_cols[2]}")

In [0]:
ax = df_weather_hourly.plot.box(column=numeric_cols[3])
ax.update_layout(title=f"Summary Statistics of {numeric_cols[3]}")

In [0]:
ax = df_weather_hourly.plot.box(column=numeric_cols[4])
ax.update_layout(title=f"Summary Statistics of {numeric_cols[4]}")

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt
df_hourly_avg_temp = (
    df_weather_hourly
    .withColumn("Hour", F.hour("Date"))
    .groupBy("Hour")
    .agg(F.avg("HourlyDryBulbTemperature").alias("AvgTemp"))
    .orderBy("Hour")
)

temperature_df = df_hourly_avg_temp.toPandas()

plt.figure(figsize=(10, 4))
sns.lineplot(data=temperature_df, x="Hour", y="AvgTemp")
plt.title("Average Temperature by Hour of Day")
plt.xlabel("Hour")
plt.ylabel("Temperature (°F)")
plt.show()

In [0]:
[
    "HourlyDryBulbTemperature", "HourlyDewPointTemperature", 
    "HourlyWindSpeed", "HourlyVisibility", "HourlyStationPressure"
][1]

In [0]:
pdf = df_weather_hourly.select(numeric_cols).toPandas()

plt.figure(figsize=(8, 6))
sns.heatmap(pdf.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap of Weather Variables")
plt.show()


# EDA for Airports and Airport Codes Dataset

In [0]:
# Airport and Airport Codes Data
df_stations = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/stations_data/stations_with_neighbors.parquet/")
df_airport_codes =  spark.read.format("csv").option("header","true").load('dbfs:/mnt/mids-w261/airport-codes_csv.csv')

In [0]:
# Num rows/cols
print(get_shape(df_stations))
print(get_shape(df_airport_codes))

null_info_df_stations = get_null_info(df_stations)
null_info_df_airport_codes = get_null_info(df_airport_codes)

In [0]:
df_stations.printSchema()
# name_df1.select("colname").distinct()

In [0]:
df_airport_codes.printSchema()

In [0]:
# Check for duplicate records in airport and airport codes dataset
print(df_stations.count(), df_stations.dropDuplicates().count())
print(df_airport_codes.count(), df_airport_codes.dropDuplicates().count())

In [0]:
display(df_stations)

In [0]:
display(df_airport_codes)

# EDA for Flights Dataset

In [0]:
df_flights = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data_3m/")
display(df_flights)

In [0]:
print(get_shape(df_flights))

In [0]:
df_flights.printSchema()

In [0]:
# todo:
# eda for origin + destiation
# eda for fl_date
# eda for dep time and dep delay
# eda for arr time and arr delay
# year
# delay by carrier (also proportionally)
# delay by origin/destination (also proportionally)
# delay by month/day of week (also proportionally)
# delay by year (also proportionally)

In [0]:
# flights by carrier
carrier_counts = (
    df_flights
    .groupBy("OP_UNIQUE_CARRIER")
    .count()
    .orderBy(F.desc("count"))
)

display(carrier_counts)

# total distinct carriers
distinct_carriers = df_flights.select("OP_UNIQUE_CARRIER").distinct().count()
print("Total distinct carriers:", distinct_carriers)

# convert to pandas for plotting
pdf = carrier_counts.toPandas()

plt.figure()
plt.bar(pdf["OP_UNIQUE_CARRIER"], pdf["count"])
plt.title("Number of Flights by Carrier")
plt.xlabel("Carrier")
plt.ylabel("Number of Flights")
plt.xticks(rotation=45)
plt.show()

In [0]:
origin_counts = (
    df_flights
    .groupBy("ORIGIN")
    .count()
    .orderBy(F.desc("count"))
)

pdf = origin_counts.limit(20).toPandas()

plt.figure()
plt.bar(pdf["ORIGIN"], pdf["count"])
plt.xticks(rotation=90)
plt.title("Top 20 Origin Airports")
plt.xlabel("Airport")
plt.ylabel("Flights")
plt.show()

In [0]:
dest_counts = (
    df_flights
    .groupBy("DEST")
    .count()
    .orderBy(F.desc("count"))
)

pdf = dest_counts.limit(20).toPandas()

plt.figure()
plt.bar(pdf["DEST"], pdf["count"])
plt.xticks(rotation=90)
plt.title("Top 20 Destination Airports")
plt.xlabel("Airport")
plt.ylabel("Flights")
plt.show()

In [0]:
dep_delay_hour = (
    df_flights
    .withColumn("DEP_HOUR", (F.col("CRS_DEP_TIME")/100).cast("int"))
    .groupBy("DEP_HOUR")
    .agg(F.avg("DEP_DELAY").alias("avg_dep_delay"))
    .orderBy("DEP_HOUR")
)

pdf = dep_delay_hour.toPandas()

plt.figure()
plt.plot(pdf["DEP_HOUR"], pdf["avg_dep_delay"])
plt.title("Average Departure Delay by Hour")
plt.xlabel("Hour")
plt.ylabel("Avg Delay (minutes)")
plt.show()

In [0]:
arr_delay_hour = (
    df_flights
    .withColumn("ARR_HOUR", (F.col("CRS_ARR_TIME")/100).cast("int"))
    .groupBy("ARR_HOUR")
    .agg(F.avg("ARR_DELAY").alias("avg_arr_delay"))
    .orderBy("ARR_HOUR")
)

pdf = arr_delay_hour.toPandas()

plt.figure()
plt.plot(pdf["ARR_HOUR"], pdf["avg_arr_delay"])
plt.title("Average Arrival Delay by Hour")
plt.xlabel("Hour")
plt.ylabel("Avg Delay (minutes)")
plt.show()

In [0]:
carrier_delay = (
    df_flights
    .groupBy("OP_UNIQUE_CARRIER")
    .agg(F.avg("DEP_DELAY").alias("avg_delay"))
    .orderBy(F.desc("avg_delay"))
)

pdf = carrier_delay.toPandas()

plt.figure()
plt.bar(pdf["OP_UNIQUE_CARRIER"], pdf["avg_delay"])
plt.title("Average Delay by Carrier")
plt.xlabel("Carrier")
plt.ylabel("Avg Delay (minutes)")
plt.show()

In [0]:
carrier_delay_rate = (
    df_flights
    .groupBy("OP_UNIQUE_CARRIER")
    .agg(F.avg("DEP_DEL15").alias("delay_rate"))
    .orderBy(F.desc("delay_rate"))
)

pdf = carrier_delay_rate.toPandas()

plt.figure()
plt.bar(pdf["OP_UNIQUE_CARRIER"], pdf["delay_rate"])
plt.title("Proportion of Delayed Flights by Carrier")
plt.xlabel("Carrier")
plt.ylabel("Delay Rate")
plt.show()

In [0]:
origin_delay = (
    df_flights
    .groupBy("ORIGIN")
    .agg(F.avg("DEP_DEL15").alias("delay_rate"))
    .orderBy(F.desc("delay_rate"))
)

pdf = origin_delay.limit(20).toPandas()

plt.figure()
plt.bar(pdf["ORIGIN"], pdf["delay_rate"])
plt.xticks(rotation=90)
plt.title("Delay Rate by Origin Airport")
plt.show()

In [0]:
delay_by_month = (
    df_flights
    .groupBy("MONTH")
    .agg(
        F.avg("DEP_DELAY").alias("avg_delay"),
        F.avg("DEP_DEL15").alias("delay_rate")
    )
    .orderBy("MONTH")
)

pdf = delay_by_month.toPandas()

plt.figure()
plt.plot(pdf["MONTH"], pdf["delay_rate"])
plt.title("Delay Rate by Month")
plt.xlabel("Month")
plt.ylabel("Delay Rate")
plt.show()

In [0]:
delay_by_dow = (
    df_flights
    .groupBy("DAY_OF_WEEK")
    .agg(
        F.avg("DEP_DELAY").alias("avg_delay"),
        F.avg("DEP_DEL15").alias("delay_rate")
    )
    .orderBy("DAY_OF_WEEK")
)

pdf = delay_by_dow.toPandas()

plt.figure()
plt.bar(pdf["DAY_OF_WEEK"], pdf["delay_rate"])
plt.title("Delay Rate by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Delay Rate")
plt.show()

# ML Algorithm Discussion

# Checkpoint data

In [0]:
# Create folder
section = "03"
number = "02"
folder_path = f"dbfs:/student-groups/Group_{section}_{number}"
# dbutils.fs.mkdirs(folder_path)

# Save hourly weather data as a parquet file
#df_weather_hourly.write.parquet(f"{folder_path}/df_weather_hourly.parquet")

OTPW Data: This is our joined data (We joined Airlines and Weather). This is the main dataset for your project, the previous 3 are given for reference. You can attempt your own join for Extra Credit. Location dbfs:/mnt/mids-w261/OTPW_60M/OTPW_60M/ and more, several samples are given!

In [0]:
# OTPW: Airline on-time performance with weather data
df_otpw = spark.read.format("csv").option("header","true").load(f"dbfs:/mnt/mids-w261/OTPW_3M_2015.csv")
display(df_otpw)

## Ensure there are no duplicate rows in OTPW data

In [0]:
assert(df_otpw.count() == df_otpw.dropDuplicates().count())

## Schema and columns of otpw data

In [0]:
df_otpw.printSchema()

In [0]:
df_otpw.columns

## Count of null values in OTPW data

In [0]:
display(df_otpw.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df_otpw.columns]))